# Människa-i-Loop: Förhandsgrindar, Risknivåindelning och Revisionsloggning

README för denna lektion introducerar Människa-i-Loop med ett kort utdrag som frågar användaren `GODKÄNN` eller `AVVISA` efter att agenten redan har producerat ett svar. Det mönstret är en bra startpunkt, men produktionens HITL-implementationer behöver vanligtvis tre ytterligare delar:

1. En **förhandsgrind** som körs **innan** agenten utför ett riskabelt steg, så att kostnad, irreversibilitet och latens hålls under kontroll.
2. **Risknivåindelning**, så att lågriskåtgärder auto-exekveras, medelriskåtgärder batchgodkänns, och endast högriskåtgärder kräver mänsklig inblandning.
3. En **revisionslogg plus revideringsloop**, så varje grind-beslut loggas som JSONL, och ett avvisande ger en ny uppmaning till agenten med en strukturerad anledning istället för att bara skriva ut `Reviderar...`.

Denna notebook bygger varje del ovanpå samma grundläggande verktyg som `06-system-message-framework.ipynb`. Den körs end-to-end i `DEMO_MODE = True` (ingen interaktiv input krävs) eller med riktiga `input()`-prompter när `DEMO_MODE = False`. Notera: i DEMO_MODE är tredje målets omförsök skriptat så att loopmekaniken är synlig end-to-end. Riktig revideringsstyrd omklassificering kräver `DEMO_MODE = False` och en operatör.

**Utanför omfattning (hanteras i andra lektioner):** autentisering och åtkomstkontroll (Lektion 06 README hot 2), middleware för verktygsanrop (Lektion 14 MAF djupdykning), fleragent-debattmönster.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Mönster 1: Förhandsgrind

README:s HITL-exempel anropar agenten först, sedan ber användaren godkänna resultatet. Det är ett **efteråtgärds**flöde. Agenten har redan kört, så kostnaden för LLM-anropet är redan betald och alla bieffekter (skickat e-post, skrivit databasrad, postat kommentar) har redan inträffat.

Ett **förhandsgrind**-flöde sätter grinden innan agenten kör det riskfyllda steget. Agenten föreslår åtgärden, grinden avgör om den ska genomföras, och endast vid godkännande sker bieffekten.

| Aspekt | Godkännande efter åtgärd (README-exempel) | Förhandsgrind (denna notebook) |
|---|---|---|
| När sker godkännandet? | Efter att agenten producerat resultat | Innan någon bieffekt körs |
| LLM-kostnad vid avslag | Redan betald | Betalas endast för förslaget, inte åtgärden |
| Oåterkalleliga bieffekter | Möjliga (åtgärden har redan skett) | Förhindrade |
| Revisionsklarhet | Godkännande är en utskrift | Godkännande är en JSON-post med tidsstämpel, åtgärd, orsak |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Mönster 2: Riskindelning

Inte varje åtgärd kräver mänskligt godkännande. En skrivskyddad sökning mot ett offentligt API har andra insatser än att skicka ett kundmail. Att behandla båda på samma sätt slösar operatörens uppmärksamhet och saktar ner agenten.

En enkel 3-nivåmodell:

| Nivå | Exempel | Godkännandeprocess |
|---|---|---|
| `low` (skrivskyddad) | Söka i en kunskapsbas, kolla flygalternativ, hämta en offentlig webbsida | Automatiskt utförande, loggat för granskning |
| `medium` (billig förändring) | Cachera ett resultat, öka en räknare, schemalägga en påminnelse | Automatiskt utförande, men samlat daglig granskning |
| `high` (extern riktad eller oåterkallelig) | Skicka ett mail, debitera ett kort, posta i en offentlig kanal | Blockera tills mänskligt godkännande |

Detta är en indelningsmetod. Produktionssystem använder ofta mer granulära nivåer (t.ex. AWS IAM-behörighetsnivåer, rollbaserade åtkomstnivåer). Den 3-nivås versionen nedan är den minsta användbara för en agent som blandar skrivskyddade och sidverkande handlingar.

Klassificeraren nedan använder nyckelordsheuristik så att demon förblir deterministisk och billig. I ett produktionssystem skulle du byta ut detta mot en inlärd klassificerare eller en policy-motor.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Mönster 3: Revisionslogg och granskningsloop

En `print("Response approved.")` är inte en revisionslogg. För förtroende bör varje grindbeslut registreras som en strukturerad händelse som du senare kan fråga, spela upp eller koppla till en incidentgranskning.

Två delar:

1. **Endast tilläggs-JSONL.** En rad per beslut, med tidsstämpel, åtgärd, nivå, beslut, orsak. Lätt att söka i med grep, lätt att skicka till en riktig logglagring senare.
2. **Revisionsloop vid avslag.** När grinden returnerar `deny`, uppmanar agenten sig själv igen med avslagsorsaken i kontext, så att nästa förslag kan undvika problemet.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Ytterligare resurser

Flera andra offentliga projekt implementerar variationer av dessa HITL-mönster. Jämför metoder för att hitta vad som passar din stack:

- **LangChain** verktygsomslag för human-in-the-loop ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): drop-in verktygsomslag som pausar exekveringen för mänsklig inmatning.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ omstrukturerade detta): använder en agentroll specifikt för att representera människan i multi-agent konversationer.
- **Semantic Kernel** funktionsfilter ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): middleware som körs runt varje funktionsanrop, lämplig för grindlogik.

Varje projekt hanterar de tre delmönstren olika: LangChain omsluter dem som verktyg, AutoGen använder en agentroll, Semantic Kernel använder middleware-filter. Läs igenom en eller två implementeringar från början till slut innan du väljer en design för din egen agent.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
